# Oracle RAG Agents: Zero to HeroBuild a complete RAG pipeline and agentic system with Oracle AI Database, OpenAI, and the Agents SDK.

## What You Will LearnIn this notebook, you will learn how to:- Set up Oracle AI Database locally for AI applications- Load, prepare, and embed research paper data- Implement 5 retrieval strategies (keyword, vector, hybrid pre/post filter, RRF, graph)- Build an end-to-end RAG pipeline- Create AI agents with tool access to Oracle retrieval- Orchestrate multi-agent systems- Add persistent session memory backed by Oracle

## Prerequisites- **OpenAI API Key** — needed for Parts 5-8- **Docker** — Oracle AI Database runs as a containerIf running in GitHub Codespaces, dependencies are pre-installed.

In [ ]:
! pip install -Uq oracledb pandas sentence-transformers datasets einops "numpy<2.0"

# Part 1: Oracle AI Database Setup**Oracle AI Database** is a converged database built for AI developers. It unifies relational, document, graph, and vector data with native support for `VECTOR` columns, HNSW indexes, `VECTOR_DISTANCE()`, Oracle Text, and SQL Property Graphs.For this workshop we use a local Docker installation of Oracle AI Database Free.

## 1.1 Docker SetupRun the following in your terminal to start Oracle:```bashdocker run -d --name oracle-free \  -p 1521:1521 \  -e ORACLE_PASSWORD=OraclePwd_2025 \  -e APP_USER=VECTOR \  -e APP_USER_PASSWORD=VectorPwd_2025 \  gvenzl/oracle-free:23-slim```Wait 2-3 minutes for the database to be ready.> If running in Codespaces, Oracle is already started for you.

### 1.1.1 Connecting to Oracle AI Database> **Companion guide:** See `docs/part-1-oracle-setup.md` for detailed explanation and troubleshooting.

In [ ]:
import os
import oracledb
import time

# ╔══════════════════════════════════════════════════════════════╗
# ║  TODO: Implement connect_to_oracle                          ║
# ║                                                              ║
# ║  1. Use oracledb.connect() with:                            ║
# ║     user="VECTOR", password="VectorPwd_2025",               ║
# ║     dsn="localhost:1521/FREEPDB1"                            ║
# ║  2. Add retry logic (max_retries attempts, retry_delay sec)  ║
# ║  3. Print the Oracle version banner on success               ║
# ║  4. Return the connection object                             ║
# ║                                                              ║
# ║  See: docs/part-1-oracle-setup.md                           ║
# ╚══════════════════════════════════════════════════════════════╝
def connect_to_oracle(max_retries=3, retry_delay=5):
    """Connect to Oracle database with retry logic."""
    pass  # YOUR CODE HERE

conn = connect_to_oracle()

# Part 2: Data Loading, Preparation & Embedding Generation> **Companion guide:** See `docs/part-2-data-loading.md`

## 2.1 Data Loading From Hugging FaceWe stream 1,000 ArXiv papers using `streaming=True` to avoid downloading the full dataset.

In [ ]:
import pandas as pdfrom datasets import load_datasetds_stream = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)

In [ ]:
sampled = []for i, item in enumerate(ds_stream):    if i >= 1000:        break    arxiv_id = item.get("arxiv_id", f"unknown_{i}")    title = item.get("title", "").strip()    abstract = item.get("abstract", "").strip()    authors = item.get("authors", [])    if isinstance(authors, str):        authors = [a.strip() for a in authors.split(",") if a.strip()]    elif isinstance(authors, list):        authors = [str(a).strip() for a in authors if str(a).strip()]    else:        authors = []    text = f"{title} — {abstract}"    sampled.append({        "id": item.get("id", f"arxiv_{i}"),        "arxiv_id": arxiv_id,        "title": title,        "abstract": abstract,        "authors": authors,        "text": text    })print(f"Streamed {len(sampled)} papers.")

In [ ]:
dataset_df = pd.DataFrame(sampled)print(f"Loaded {len(dataset_df)} rows into DataFrame.")dataset_df.head()

## 2.2 Embedding GenerationWe use **nomic-ai/nomic-embed-text-v1.5** (768-dim, local, no API key needed).Nomic uses prefix-based embeddings:- `search_document:` for documents- `search_query:` for queries

In [ ]:
from sentence_transformers import SentenceTransformerembedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

In [ ]:
import numpy as npimport sysdataset_df["text_prefixed"] = dataset_df["text"].apply(lambda x: f"search_document: {x}")texts = dataset_df["text_prefixed"].tolist()embs = []total = len(texts)for i, text in enumerate(texts, start=1):    embedding = embedding_model.encode(        text, show_progress_bar=False,        convert_to_numpy=True, normalize_embeddings=True    )    embs.append(embedding)    sys.stdout.write(f"\rEncoding {i}/{total} texts...")    sys.stdout.flush()print(f"\nFinished encoding {len(embs)} texts.")embs = np.vstack(embs)print(f"Embeddings shape: {embs.shape}")

In [ ]:
dataset_df["embedding"] = [e.astype(np.float32).tolist() for e in embs]dim = len(dataset_df["embedding"].iloc[0])print(f"Embedding dimension: {dim}")dataset_df.head(2)

# Part 3: Database Table Setup & Data Ingestion> **Companion guide:** See `docs/part-3-table-setup.md`

## 3.1 Create Research Papers Table

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗# ║  TODO: Write the DDL to create the research_papers table    ║# ║                                                              ║# ║  1. Safely drop dependent tables first (paper_similarities,  ║# ║     paper_authors, authors, research_papers)                 ║# ║  2. Create research_papers with columns:                     ║# ║     - arxiv_id VARCHAR2(255) PRIMARY KEY                     ║# ║     - title VARCHAR2(4000)                                   ║# ║     - abstract VARCHAR2(4000)                                ║# ║     - text CLOB                                              ║# ║     - embedding VECTOR({dim}, FLOAT32)                     ║# ║                                                              ║# ║  Use BEGIN/EXCEPTION blocks for safe drops (SQLCODE -942)    ║# ║  Separate statements with /                                  ║# ║                                                              ║# ║  See: docs/part-3-table-setup.md                            ║# ╚══════════════════════════════════════════════════════════════╝ddl = f"""# YOUR DDL HERE"""

In [ ]:
with conn.cursor() as cur:    for stmt in ddl.split("/"):        if stmt.strip():            cur.execute(stmt)conn.commit()print(f"Table RESEARCH_PAPERS created with VECTOR dimension: {dim}")

## 3.2 Create Indexes (Vector and Search)

In [ ]:
with conn.cursor() as cur:    for idx_name in ("RP_VEC_HNSW", "RP_VEC_IVF"):        try:            cur.execute(f"DROP INDEX {idx_name}")        except oracledb.DatabaseError as e:            if "ORA-01418" not in str(e):                raise    cur.execute("""        CREATE VECTOR INDEX RP_VEC_HNSW        ON research_papers(embedding)        ORGANIZATION INMEMORY NEIGHBOR GRAPH        DISTANCE COSINE        WITH TARGET ACCURACY 90        PARAMETERS (TYPE HNSW, NEIGHBORS 40, EFCONSTRUCTION 500)        TABLESPACE USERS    """)conn.commit()print("Vector Index RP_VEC_HNSW (HNSW) created")

In [ ]:
with conn.cursor() as cur:    try:        cur.execute("DROP INDEX rp_text_idx")    except oracledb.DatabaseError as e:        if "ORA-01418" not in str(e):            raise    cur.execute("""        CREATE INDEX rp_text_idx        ON research_papers(text)        INDEXTYPE IS CTXSYS.CONTEXT        PARAMETERS ('SYNC (ON COMMIT)')    """)conn.commit()print("Oracle Text index (rp_text_idx) created on TEXT column.")

### 3.2.1 Create Relational Tables for Graph Retrieval

In [ ]:
graph_tables_ddl = """BEGIN    EXECUTE IMMEDIATE 'DROP TABLE paper_similarities';EXCEPTION WHEN OTHERS THEN    IF SQLCODE != -942 THEN RAISE; END IF;END;/BEGIN    EXECUTE IMMEDIATE 'DROP TABLE paper_authors';EXCEPTION WHEN OTHERS THEN    IF SQLCODE != -942 THEN RAISE; END IF;END;/BEGIN    EXECUTE IMMEDIATE 'DROP TABLE authors';EXCEPTION WHEN OTHERS THEN    IF SQLCODE != -942 THEN RAISE; END IF;END;/CREATE TABLE authors (    author_name VARCHAR2(512) PRIMARY KEY)TABLESPACE USERS/CREATE TABLE paper_authors (    arxiv_id VARCHAR2(255) NOT NULL,    author_name VARCHAR2(512) NOT NULL,    CONSTRAINT pk_paper_authors PRIMARY KEY (arxiv_id, author_name),    CONSTRAINT fk_pa_paper FOREIGN KEY (arxiv_id) REFERENCES research_papers(arxiv_id),    CONSTRAINT fk_pa_author FOREIGN KEY (author_name) REFERENCES authors(author_name))TABLESPACE USERS/CREATE TABLE paper_similarities (    source_arxiv_id VARCHAR2(255) NOT NULL,    target_arxiv_id VARCHAR2(255) NOT NULL,    sim_score NUMBER(8,6) NOT NULL,    rank_no NUMBER(5) NOT NULL,    CONSTRAINT pk_paper_similarities PRIMARY KEY (source_arxiv_id, target_arxiv_id),    CONSTRAINT fk_ps_src FOREIGN KEY (source_arxiv_id) REFERENCES research_papers(arxiv_id),    CONSTRAINT fk_ps_tgt FOREIGN KEY (target_arxiv_id) REFERENCES research_papers(arxiv_id),    CONSTRAINT ck_ps_not_self CHECK (source_arxiv_id <> target_arxiv_id))TABLESPACE USERS"""with conn.cursor() as cur:    for stmt in graph_tables_ddl.split("/"):        if stmt.strip():            cur.execute(stmt)    cur.execute("CREATE INDEX idx_pa_author ON paper_authors(author_name)")    cur.execute("CREATE INDEX idx_ps_source ON paper_similarities(source_arxiv_id)")    cur.execute("CREATE INDEX idx_ps_target ON paper_similarities(target_arxiv_id)")conn.commit()print("Graph tables created: AUTHORS, PAPER_AUTHORS, PAPER_SIMILARITIES")

## 3.3 Ingest Data into Oracle

In [ ]:
from tqdm import tqdmimport arrayrows = []for i, row in dataset_df.iterrows():    embedding_array = array.array('f', row.get("embedding"))    rows.append((        row.get("arxiv_id"),        row.get("title"),        row.get("abstract"),        row.get("text"),        embedding_array    ))print(f"Preparing to insert {len(rows)} rows into RESEARCH_PAPERS...")with conn.cursor() as cur:    for row in tqdm(rows):        cur.execute("""            INSERT INTO research_papers (arxiv_id, title, abstract, text, embedding)            VALUES (:1, :2, :3, :4, :5)        """, row)conn.commit()print("Data inserted successfully")

In [ ]:
with conn.cursor() as cur:    cur.execute("SELECT COUNT(*) FROM RESEARCH_PAPERS")    print("Row count:", cur.fetchone()[0])    cur.execute("""        SELECT arxiv_id, title, abstract, text FROM RESEARCH_PAPERS        FETCH FIRST 3 ROWS ONLY    """)    for row in cur.fetchall():        print(row)

## 3.4 Build and Register Graph Relationships

### 3.4.1 Load Author Edges (`Author -> WROTE -> Paper`)

In [ ]:
def normalize_authors(raw_authors):    if raw_authors is None:        return []    if isinstance(raw_authors, str):        return [a.strip() for a in raw_authors.split(",") if a.strip()]    if isinstance(raw_authors, list):        return [str(a).strip() for a in raw_authors if str(a).strip()]    return []author_rows = set()paper_author_rows = set()for _, row in dataset_df.iterrows():    arxiv_id = row.get("arxiv_id")    if not arxiv_id:        continue    for author_name in normalize_authors(row.get("authors")):        author_rows.add((author_name,))        paper_author_rows.add((arxiv_id, author_name))with conn.cursor() as cur:    cur.execute("TRUNCATE TABLE paper_authors")    cur.execute("TRUNCATE TABLE authors")    if author_rows:        cur.executemany("INSERT INTO authors (author_name) VALUES (:1)", sorted(author_rows))    if paper_author_rows:        cur.executemany("INSERT INTO paper_authors (arxiv_id, author_name) VALUES (:1, :2)", sorted(paper_author_rows))conn.commit()print(f"Loaded {len(author_rows)} authors and {len(paper_author_rows)} Author->WROTE->Paper edges.")

### 3.4.2 Load Similarity Edges (`Paper -> SIMILAR_TO -> Paper`)

In [ ]:
TOP_SIM_NEIGHBORS = 10paper_ids = dataset_df["arxiv_id"].tolist()emb_matrix = np.vstack(dataset_df["embedding"].values).astype(np.float32)neighbor_k = min(TOP_SIM_NEIGHBORS, max(len(paper_ids) - 1, 0))similarity_rows = []if neighbor_k > 0:    similarity_matrix = emb_matrix @ emb_matrix.T    np.fill_diagonal(similarity_matrix, -np.inf)    for i, source_arxiv_id in enumerate(paper_ids):        nn_idx = np.argpartition(similarity_matrix[i], -neighbor_k)[-neighbor_k:]        nn_idx = nn_idx[np.argsort(similarity_matrix[i][nn_idx])[::-1]]        for rank_no, j in enumerate(nn_idx, start=1):            similarity_rows.append((                source_arxiv_id, paper_ids[j],                float(similarity_matrix[i][j]), rank_no            ))with conn.cursor() as cur:    cur.execute("TRUNCATE TABLE paper_similarities")    if similarity_rows:        cur.executemany("""            INSERT INTO paper_similarities (source_arxiv_id, target_arxiv_id, sim_score, rank_no)            VALUES (:1, :2, :3, :4)        """, similarity_rows)conn.commit()print(f"Loaded {len(similarity_rows)} Paper->SIMILAR_TO->Paper edges (top {neighbor_k} per paper).")

### 3.4.3 Register the Oracle SQL Property Graph

In [ ]:
with conn.cursor() as cur:    cur.execute("""        SELECT COUNT(*) FROM user_property_graphs WHERE graph_name = 'RESEARCH_GRAPH'    """)    if cur.fetchone()[0] > 0:        cur.execute("DROP PROPERTY GRAPH research_graph")    cur.execute("""        CREATE PROPERTY GRAPH research_graph        VERTEX TABLES (            research_papers KEY (arxiv_id) LABEL paper                PROPERTIES (arxiv_id, title, abstract),            authors KEY (author_name) LABEL author                PROPERTIES (author_name)        )        EDGE TABLES (            paper_authors KEY (arxiv_id, author_name)                SOURCE KEY (author_name) REFERENCES authors (author_name)                DESTINATION KEY (arxiv_id) REFERENCES research_papers (arxiv_id)                LABEL wrote,            paper_similarities KEY (source_arxiv_id, target_arxiv_id)                SOURCE KEY (source_arxiv_id) REFERENCES research_papers (arxiv_id)                DESTINATION KEY (target_arxiv_id) REFERENCES research_papers (arxiv_id)                LABEL similar_to                PROPERTIES (sim_score, rank_no)        )    """)conn.commit()print("SQL Property Graph RESEARCH_GRAPH created.")

# Part 4: Retrieval Mechanisms> **Companion guide:** See `docs/part-4-retrieval.md`

In [ ]:
SEARCH_TEXT_KEYWORDS = "optimization"

## 4.1 Text Based Retrieval

In [ ]:
def keyword_search_research_papers(conn, keyword: str):    """Perform a full-text keyword search using the Oracle Text index."""    query = """        SELECT arxiv_id, title, SUBSTR(text, 1, 200) AS text_snippet,               SCORE(1) AS relevance_score        FROM research_papers        WHERE CONTAINS(text, :keyword, 1) > 0        ORDER BY SCORE(1) DESC        FETCH FIRST 10 ROWS ONLY    """    with conn.cursor() as cur:        cur.execute(query, keyword=keyword)        rows = cur.fetchall()        columns = [desc[0] for desc in cur.description]    return rows, columns

In [ ]:
rows, columns = keyword_search_research_papers(conn, SEARCH_TEXT_KEYWORDS)results_df = pd.DataFrame(rows, columns=columns)results_df

## 4.2 Vector Based Retrieval

In [ ]:
SEARCH_QUERY = "Get me papers related to planetary exploration"

In [ ]:
import array# ╔══════════════════════════════════════════════════════════════╗# ║  TODO: Implement vector_search_research_papers               ║# ║                                                              ║# ║  1. Encode the query with "search_query:" prefix            ║# ║  2. Convert embedding to array.array('f', ...)              ║# ║  3. Run SQL with VECTOR_DISTANCE(embedding, :q, COSINE)    ║# ║  4. Use FETCH APPROX FIRST {top_k} ROWS ONLY               ║# ║     WITH TARGET ACCURACY 90                                  ║# ║  5. Return (rows, columns) tuple                             ║# ║                                                              ║# ║  See: docs/part-4-retrieval.md                              ║# ╚══════════════════════════════════════════════════════════════╝def vector_search_research_papers(conn, embedding_model, search_query: str, top_k: int = 5):    """Perform a vector similarity search on the research_papers table."""    pass  # YOUR CODE HERE

In [ ]:
rows, columns = vector_search_research_papers(conn, embedding_model, SEARCH_TEXT_KEYWORDS, top_k=5)results_df = pd.DataFrame(rows, columns=columns)results_df

## 4.3 Hybrid Retrieval (Vector + Text Combined)### 4.3.1 Pre-Filtering

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗# ║  TODO: Implement hybrid_search_research_papers_pre_filter    ║# ║                                                              ║# ║  Combine keyword filtering with vector ranking:              ║# ║  1. Encode the query (same as vector search)                 ║# ║  2. Add WHERE CONTAINS(text, :kw, 1) > 0 for pre-filtering  ║# ║  3. Rank by VECTOR_DISTANCE                                  ║# ║  4. Return (rows, columns, None) tuple                       ║# ║                                                              ║# ║  See: docs/part-4-retrieval.md                              ║# ╚══════════════════════════════════════════════════════════════╝def hybrid_search_research_papers_pre_filter(    conn, embedding_model, search_phrase: str,    top_k: int = 10, show_explain: bool = False):    """Hybrid search: keyword pre-filter + vector ranking."""    pass  # YOUR CODE HERE

In [ ]:
rows, columns, _ = hybrid_search_research_papers_pre_filter(    conn, embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS, top_k=5, show_explain=False)pre_filter_results_df = pd.DataFrame(rows, columns=columns)pre_filter_results_df

### 4.3.2 Post-Filtering

In [ ]:
def hybrid_search_research_papers_postfilter(    conn, embedding_model, search_phrase: str,    top_k: int = 10, candidate_k: int = 200, show_explain: bool = False):    """Hybrid search: vector candidates first, then text filter."""    query_embedding = embedding_model.encode(        [f"search_query: {search_phrase}"],        convert_to_numpy=True, normalize_embeddings=True    )[0].astype(np.float32).tolist()    query_embedding_array = array.array('f', query_embedding)    with conn.cursor() as cur:        sql = f"""            WITH vec_candidates AS (                SELECT arxiv_id, title, abstract, text,                       1 - VECTOR_DISTANCE(embedding, :q, COSINE) AS similarity_score                FROM research_papers                ORDER BY similarity_score DESC                FETCH APPROX FIRST {candidate_k} ROWS ONLY WITH TARGET ACCURACY 90            )            SELECT arxiv_id, title,                   SUBSTR(text, 1, 200) AS text_snippet,                   ROUND(similarity_score, 4) AS similarity_score            FROM vec_candidates            WHERE CONTAINS(text, :kw, 1) > 0            ORDER BY similarity_score DESC            FETCH FIRST {top_k} ROWS ONLY        """        cur.execute(sql, q=query_embedding_array, kw=search_phrase)        rows = cur.fetchall()        columns = [desc[0] for desc in cur.description]    return rows, columns, None

In [ ]:
rows, columns, _ = hybrid_search_research_papers_postfilter(    conn, embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS, top_k=5, candidate_k=200, show_explain=False)post_filter_results_df = pd.DataFrame(rows, columns=columns)post_filter_results_df

| Approach | Strength | Best For || --- | --- | --- || Pre-filter | Fast; reduces candidate set early | Known keywords + semantic ranking || Post-filter | Broader semantic recall first | Exploratory queries with keyword refinement |

### 4.3.3 Reciprocal Rank Fusion

In [ ]:
def hybrid_rrf_search(    conn, embedding_model, search_phrase: str,    top_k: int = 10, per_list: int = 120, k: int = 60,    phrase_safe: bool = True, show_explain: bool = False):    """RRF fusion of Vector + Oracle Text results."""    qv = embedding_model.encode(        [f"search_query: {search_phrase}"],        convert_to_numpy=True, normalize_embeddings=True    )[0].astype(np.float32).tolist()    qv = array.array('f', qv)    kw = f"\"{search_phrase}\"" if (phrase_safe and " " in search_phrase.strip()) else search_phrase    with conn.cursor() as cur:        sql = f"""            WITH            vec AS (              SELECT arxiv_id, title, SUBSTR(text, 1, 200) AS text_snippet,                     1 - VECTOR_DISTANCE(embedding, :q, COSINE) AS sim_vec,                     ROW_NUMBER() OVER (ORDER BY 1 - VECTOR_DISTANCE(embedding, :q, COSINE) DESC) AS r_vec              FROM research_papers              FETCH APPROX FIRST {per_list} ROWS ONLY WITH TARGET ACCURACY 90            ),            txt AS (              SELECT arxiv_id, title, SUBSTR(text, 1, 200) AS text_snippet,                     SCORE(1) AS score_txt,                     ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) AS r_txt              FROM research_papers              WHERE CONTAINS(text, :kw, 1) > 0              FETCH FIRST {per_list} ROWS ONLY            ),            fused AS (              SELECT COALESCE(v.arxiv_id, t.arxiv_id) AS arxiv_id,                     COALESCE(v.title, t.title) AS title,                     COALESCE(v.text_snippet, t.text_snippet) AS text_snippet,                     NVL(v.r_vec, 999999) AS r_vec,                     NVL(t.r_txt, 999999) AS r_txt,                     NVL(v.sim_vec, 0) AS sim_vec,                     NVL(t.score_txt, 0) AS score_txt              FROM vec v FULL OUTER JOIN txt t ON t.arxiv_id = v.arxiv_id            )            SELECT arxiv_id, title, text_snippet,                   ROUND((1.0/(:k + r_vec)) + (1.0/(:k + r_txt)), 6) AS rrf_score,                   r_vec, r_txt, ROUND(sim_vec, 4) AS sim_vec, ROUND(score_txt, 4) AS score_txt            FROM fused            ORDER BY rrf_score DESC, title            FETCH FIRST {top_k} ROWS ONLY        """        cur.execute(sql, q=qv, kw=kw, k=k)        rows = cur.fetchall()        columns = [d[0] for d in cur.description]    return rows, columns, None

In [ ]:
rows, columns, _ = hybrid_rrf_search(    conn, embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS,    top_k=5, per_list=60, k=60, show_explain=False)rrf_results_df = pd.DataFrame(rows, columns=columns)rrf_results_df

## 4.4 Graph-Based Retrieval (Oracle SQL Property Graph)Seeds with vector similarity, then expands via SIMILAR_TO and shared-author paths using `GRAPH_TABLE()`.

In [ ]:
def graph_search_research_papers(    conn, embedding_model, search_query: str, top_k: int = 10, seed_k: int = 25):    """Graph retrieval using Oracle SQL Property Graph + GRAPH_TABLE."""    seed_k = max(seed_k, top_k)    query_embedding = embedding_model.encode(        [f"search_query: {search_query}"],        convert_to_numpy=True, normalize_embeddings=True    )[0].astype(np.float32).tolist()    query_embedding_array = array.array('f', query_embedding)    sql = f"""        WITH seed AS (            SELECT arxiv_id, 1 - VECTOR_DISTANCE(embedding, :q, COSINE) AS seed_score            FROM research_papers            ORDER BY seed_score DESC            FETCH APPROX FIRST {seed_k} ROWS ONLY WITH TARGET ACCURACY 90        ),        seed_hits AS (            SELECT arxiv_id AS source_arxiv_id, arxiv_id AS candidate_arxiv_id,                   seed_score, 'seed' AS relation_type, seed_score AS edge_score            FROM seed        ),        sim_hops AS (            SELECT s.arxiv_id AS source_arxiv_id, gt.target_arxiv_id AS candidate_arxiv_id,                   s.seed_score, 'similar_to' AS relation_type, gt.edge_score AS edge_score            FROM seed s            JOIN GRAPH_TABLE(                research_graph                MATCH (src IS paper)-[e IS similar_to]->(dst IS paper)                COLUMNS (src.arxiv_id AS source_arxiv_id, dst.arxiv_id AS target_arxiv_id, e.sim_score AS edge_score)            ) gt ON gt.source_arxiv_id = s.arxiv_id        ),        author_hops AS (            SELECT s.arxiv_id AS source_arxiv_id, gt.target_arxiv_id AS candidate_arxiv_id,                   s.seed_score, 'shared_author' AS relation_type, 1.0 AS edge_score            FROM seed s            JOIN GRAPH_TABLE(                research_graph                MATCH (src IS paper)<-[w1 IS wrote]-(a IS author)-[w2 IS wrote]->(dst IS paper)                COLUMNS (src.arxiv_id AS source_arxiv_id, dst.arxiv_id AS target_arxiv_id)            ) gt ON gt.source_arxiv_id = s.arxiv_id            WHERE gt.target_arxiv_id <> s.arxiv_id        ),        candidates AS (            SELECT * FROM seed_hits UNION ALL            SELECT * FROM sim_hops UNION ALL            SELECT * FROM author_hops        ),        scored AS (            SELECT candidate_arxiv_id AS arxiv_id,                   MAX(CASE relation_type                       WHEN 'seed' THEN seed_score                       WHEN 'similar_to' THEN (0.70 * seed_score) + (0.30 * edge_score)                       WHEN 'shared_author' THEN (0.85 * seed_score) + (0.15 * edge_score)                       ELSE seed_score END) AS graph_score            FROM candidates GROUP BY candidate_arxiv_id        )        SELECT rp.arxiv_id, rp.title, rp.abstract,               SUBSTR(rp.text, 1, 200) AS text_snippet,               ROUND(sc.graph_score, 4) AS graph_score        FROM scored sc JOIN research_papers rp ON rp.arxiv_id = sc.arxiv_id        ORDER BY graph_score DESC        FETCH FIRST {top_k} ROWS ONLY    """    with conn.cursor() as cur:        cur.execute(sql, q=query_embedding_array)        rows = cur.fetchall()        columns = [desc[0] for desc in cur.description]    return rows, columns

In [ ]:
rows, columns = graph_search_research_papers(    conn, embedding_model, search_query=SEARCH_TEXT_KEYWORDS, top_k=5, seed_k=20)graph_results_df = pd.DataFrame(rows, columns=columns)graph_results_df

### 4.5 Compare Top Results Across Retrieval Strategies

In [ ]:
def extract_top_titles(rows, columns, k=5):    """Return top-k titles from retrieval results."""    titles = []    for row in rows[:k]:        row_data = dict(zip(columns, row))        titles.append(row_data.get("TITLE", "Untitled"))    return titlescomparison_specs = [    ("keyword", lambda: keyword_search_research_papers(conn, SEARCH_TEXT_KEYWORDS)),    ("vector", lambda: vector_search_research_papers(conn, embedding_model, SEARCH_TEXT_KEYWORDS, top_k=5)),    ("hybrid_pre_filter", lambda: hybrid_search_research_papers_pre_filter(        conn=conn, embedding_model=embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS, top_k=5, show_explain=False)),    ("hybrid_postfilter", lambda: hybrid_search_research_papers_postfilter(        conn=conn, embedding_model=embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS, top_k=5, candidate_k=200, show_explain=False)),    ("hybrid_rrf", lambda: hybrid_rrf_search(        conn=conn, embedding_model=embedding_model, search_phrase=SEARCH_TEXT_KEYWORDS, top_k=5, per_list=60, k=60, show_explain=False)),    ("graph", lambda: graph_search_research_papers(        conn=conn, embedding_model=embedding_model, search_query=SEARCH_TEXT_KEYWORDS, top_k=5, seed_k=20)),]comparison_rows = []for strategy_name, runner in comparison_specs:    result = runner()    if isinstance(result, tuple) and len(result) >= 2:        rows, columns = result[0], result[1]    else:        rows, columns = [], []    titles = extract_top_titles(rows, columns, k=5)    record = {"retrieval_strategy": strategy_name}    for i in range(5):        record[f"Top_{i+1}"] = titles[i] if i < len(titles) else ""    comparison_rows.append(record)retrieval_strategy_comparison_df = pd.DataFrame(comparison_rows)retrieval_strategy_comparison_df

# Part 5: Building a RAG Pipeline> **Companion guide:** See `docs/part-5-rag-pipeline.md`Flow:1. Configure API access and initialize the OpenAI client2. Define a reusable RAG function supporting multiple retrieval modes3. Run an end-to-end query

### 5.1 Configure API Access

In [ ]:
import getpassimport osdef set_env_securely(var_name, prompt):    value = getpass.getpass(prompt)    os.environ[var_name] = value

In [ ]:
set_env_securely("OPENAI_API_KEY", "Enter your OpenAI API Key: ")

### 5.2 Initialize and Smoke-Test the OpenAI Client

In [ ]:
from openai import OpenAIOPENAI_MODEL = "gpt-4o"openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))response = openai_client.responses.create(    model=OPENAI_MODEL,    input="Hello! I'm a user!",    instructions="You are a research paper assistant.",)print(response.output_text)

### 5.3 Define the Reusable RAG Function

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗# ║  TODO: Implement research_paper_assistant_rag_pipeline       ║# ║                                                              ║# ║  1. Route to the correct retrieval function based on         ║# ║     retrieval_mode (keyword, vector, graph, or hybrid)       ║# ║  2. Format retrieved rows into citation-ready context        ║# ║     with [1], [2], etc.                                      ║# ║  3. Construct a prompt with the user query and context       ║# ║  4. Call openai_client.responses.create() with the prompt    ║# ║  5. Return response.output_text                              ║# ║                                                              ║# ║  See: docs/part-5-rag-pipeline.md                           ║# ╚══════════════════════════════════════════════════════════════╝def research_paper_assistant_rag_pipeline(    conn, embedding_model, user_query: str,    top_k: int = 10, retrieval_mode: str = "hybrid", show_explain: bool = False):    """RAG pipeline: retrieve from Oracle, format context, call LLM."""    pass  # YOUR CODE HERE

### 5.4 Run an End-to-End RAG Example

In [ ]:
summary = research_paper_assistant_rag_pipeline(    conn=conn, embedding_model=embedding_model,    user_query=SEARCH_TEXT_KEYWORDS, top_k=10,    retrieval_mode="hybrid", show_explain=False)print(summary)

# Part 6: AI Agents with OpenAI and Oracle AI Database> **Companion guide:** See `docs/part-6-agents-basics.md`In this section, we move from a single RAG call to an agentic system:1. Start with a base agent2. Add retrieval tools backed by Oracle SQL3. Upgrade to multi-tool routing and orchestration4. Add conversation history and persistent session memory

### 6.1 Install Agent Runtime

In [ ]:
%pip install -Uq --no-cache-dir openai openai-agents

In [ ]:
import sysimport importlib.metadata as md_pkgprint("Python executable:", sys.executable)print("openai version:", md_pkg.version("openai"))print("openai-agents version:", md_pkg.version("openai-agents"))

### 6.2 Create a Baseline Agent (No Tools Yet)

In [ ]:
from agents import Agent, Runnerresearch_paper_assistant = Agent(    name="Research Paper Assistant",    model=OPENAI_MODEL,    instructions="""      You are a Research Paper Assistant focused on helping users explore, analyze, and summarize      academic research.      Maintain a professional, concise, and scholarly tone appropriate for research discussions.    """,)

In [ ]:
run_result = await Runner.run(    starting_agent=research_paper_assistant,    input="Summarize recent research on optimization algorithms",)print(run_result.final_output)

### 6.3 Expose Oracle Retrieval as a Callable Tool

In [ ]:
from agents.tool import function_tool# ╔══════════════════════════════════════════════════════════════╗# ║  TODO: Implement get_research_papers tool                    ║# ║                                                              ║# ║  Use @function_tool decorator to wrap retrieval as a tool    ║# ║  1. Accept user_query, retrieval_mode, top_k                 ║# ║  2. Route to correct retrieval function (keyword, vector,    ║# ║     graph, or hybrid)                                        ║# ║  3. Format results with numbered citations [1], [2], etc.    ║# ║  4. Return formatted string                                  ║# ║                                                              ║# ║  See: docs/part-6-agents-basics.md                          ║# ╚══════════════════════════════════════════════════════════════╝@function_tooldef get_research_papers(user_query: str, retrieval_mode: str = "hybrid", top_k: int = 5) -> str:    """Retrieves academic research papers relevant to the user's query."""    pass  # YOUR CODE HERE

In [ ]:
research_paper_assistant.tools.append(get_research_papers)

In [ ]:
run_result_with_tool = await Runner.run(    starting_agent=research_paper_assistant,    input="Get me information on reinforcement learning research papers",)print(run_result_with_tool.final_output)

In [ ]:
import pprintpprint.pprint(run_result_with_tool.raw_responses)

### 6.4 Add a Second Tool and Enable Multi-Tool Access

In [ ]:
@function_tooldef get_past_research_conversations(user_query: str, top_k: int = 5) -> str:    """Retrieves relevant past research-related conversations or analyses.    Args:        user_query: The research topic or question to search for.        top_k: Number of top past discussions to retrieve (default=5).    Returns:        Formatted examples of relevant past research discussions.    """    rows, columns, _ = hybrid_search_research_papers_pre_filter(        conn=conn, embedding_model=embedding_model,        search_phrase=user_query, top_k=top_k, show_explain=False    )    retrieved_count = len(rows) if rows else 0    if retrieved_count == 0:        return f"No past research discussions found related to '{user_query}'."    formatted_results = [f"{retrieved_count} past research discussions for: '{user_query}'\n"]    for i, row in enumerate(rows):        row_data = dict(zip(columns, row))        title = row_data.get("TITLE", "Untitled Discussion")        abstract = row_data.get("ABSTRACT", "No summary available.")        snippet = row_data.get("TEXT_SNIPPET", "")        score = row_data.get("SIMILARITY_SCORE") or row_data.get("TEXT_RELEVANCE_SCORE") or "N/A"        formatted_results.append(            f"[{i+1}] **{title}**\nSummary: {abstract}\nSnippet: {snippet}\nRelevance: {score}\n"        )    return "\n".join(formatted_results)

### 6.5 Strengthen Agent Instructions for Tool Routing

In [ ]:
upgraded_research_paper_assistant = Agent(    name="Research Paper Assistant",    model=OPENAI_MODEL,    instructions="""    Always maintain an academic, evidence-based tone.    Your purpose is to help users explore, synthesize, and connect research insights —    not to speculate or fabricate information.    """,)

In [ ]:
upgraded_research_paper_assistant.tools.append(get_research_papers)upgraded_research_paper_assistant.tools.append(get_past_research_conversations)pprint.pprint(upgraded_research_paper_assistant.tools)

In [ ]:
run_result_with_tools = await Runner.run(    starting_agent=upgraded_research_paper_assistant,    input=(        "Get me information on reinforcement learning research papers "        "and also check if there were any past discussions on this topic"    ),)print(run_result_with_tools.final_output)

## Agent-as-Tools Orchestration> **Companion guide:** See `docs/part-7-orchestration.md`Now we compose specialists:- a research-paper retrieval specialist- a past-conversation retrieval specialist- an orchestrator that delegates- a final synthesizer that produces a cohesive response

### Orchestration Flow1. Define specialized agents with clear scope2. Register them as tools on an orchestrator agent3. Run the orchestrator to gather relevant evidence4. Pass gathered evidence to a synthesizer for final answer generation

In [ ]:
research_paper_agent = Agent(    name="research_paper_agent",    instructions="""        You specialize in retrieving and summarizing academic research papers.        Use the get_research_papers tool to find relevant literature.        Always cite sources using [1], [2], etc.    """,    handoff_description="A research retrieval specialist with access to academic papers.",    tools=[get_research_papers],)research_conversation_agent = Agent(    name="research_conversation_agent",    instructions="""        You specialize in retrieving and summarizing past research discussions.        Use the get_past_research_conversations tool to surface relevant prior sessions.    """,    handoff_description="A research memory specialist with access to prior discussions.",    tools=[get_past_research_conversations],)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗# ║  TODO: Implement the orchestrator_agent                      ║# ║                                                              ║# ║  Use agent.as_tool() to register specialists:                ║# ║  1. research_paper_agent.as_tool(                            ║# ║       tool_name="translate_to_research_papers", ...)         ║# ║  2. research_conversation_agent.as_tool(                     ║# ║       tool_name="translate_to_research_conversations", ...)  ║# ║                                                              ║# ║  Write clear instructions for when to use each tool.         ║# ║                                                              ║# ║  See: docs/part-7-orchestration.md                          ║# ╚══════════════════════════════════════════════════════════════╝orchestrator_agent = Agent(    name="research_assistant_orchestrator",    instructions="...",  # YOUR INSTRUCTIONS HERE    tools=[        # YOUR TOOLS HERE    ],)

In [ ]:
synthesizer_agent = Agent(    name="research_response_synthesizer",    instructions=(        "You create comprehensive, well-organized research summaries.\n\n"        "1) Start with a concise overview (3-5 sentences).\n"        "2) Separate NEW LITERATURE from PAST DISCUSSIONS.\n"        "3) Cite sources using [1], [2], etc.\n"        "4) Emphasize methods, evidence, and limitations.\n"        "5) Conclude with open questions and gaps.\n"        "Tone: academic, objective, and precise."    ),)

### 6.6 Run the Asynchronous Orchestration Workflow

In [ ]:
from agents import ItemHelpers, MessageOutputItem, traceasync def research_assistant_workflow(user_query: str):    """Run the complete research assistant workflow."""    with trace("Research Orchestrator"):        orchestrator_result = await Runner.run(orchestrator_agent, user_query)        print("\n--- Research Orchestration Steps ---")        for item in orchestrator_result.new_items:            if isinstance(item, MessageOutputItem):                text = ItemHelpers.text_message_output(item)                if text:                    print(f"  - Retrieval step: {text}")        synthesizer_result = await Runner.run(            synthesizer_agent, orchestrator_result.to_input_list()        )        print(f"\n\n--- Final Research Synthesis ---\n{synthesizer_result.final_output}\n")    return synthesizer_result.final_output

In [ ]:
import asyncioimport nest_asyncionest_asyncio.apply()

In [ ]:
def run_virtual_research_assistant(query):    loop = asyncio.new_event_loop()    asyncio.set_event_loop(loop)    result = loop.run_until_complete(research_assistant_workflow(query))    loop.close()    return result

In [ ]:
query = input("What research topic can I help you with today? ")run_virtual_research_assistant(query)

## Agentic Chat SystemThe next block adds thread-aware conversation handling on top of orchestration.

In [ ]:
import datetimeimport uuidwith conn.cursor() as cur:    cur.execute("""        BEGIN            EXECUTE IMMEDIATE 'DROP TABLE chat_history';        EXCEPTION WHEN OTHERS THEN            IF SQLCODE != -942 THEN RAISE; END IF;        END;    """)    cur.execute("""        CREATE TABLE chat_history (            id VARCHAR2(100) PRIMARY KEY,            thread_id VARCHAR2(100) NOT NULL,            role VARCHAR2(20) NOT NULL,            message CLOB NOT NULL,            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP        )        TABLESPACE USERS    """)    cur.execute("""        CREATE INDEX idx_thread_timestamp        ON chat_history(thread_id, timestamp)        TABLESPACE USERS    """)    conn.commit()    print("Table chat_history created successfully with index.")

In [ ]:
async def research_assistant_chat(user_query, thread_id=None):    """Run the research assistant with conversation history."""    if thread_id is None:        thread_id = str(uuid.uuid4())        print(f"New conversation started with thread ID: {thread_id}")    else:        print(f"Continuing conversation with thread ID: {thread_id}")    # Store user query    message_id = str(uuid.uuid4())    with conn.cursor() as cur:        cur.execute("""            INSERT INTO chat_history (id, thread_id, role, message, timestamp)            VALUES (:id, :thread_id, :role, :message, CURRENT_TIMESTAMP)        """, {'id': message_id, 'thread_id': thread_id, 'role': 'user', 'message': user_query})        conn.commit()    # Retrieve history    with conn.cursor() as cur:        cur.execute("""            SELECT role, message, timestamp FROM chat_history            WHERE thread_id = :thread_id ORDER BY timestamp ASC        """, {'thread_id': thread_id})        chat_history = cur.fetchall()    conversation_context = ""    for role, message, timestamp in chat_history:        conversation_context += f"{'User' if role == 'user' else 'Assistant'}: {message}\n"    # Run orchestration    with trace("Research Orchestrator"):        orchestrator_result = await Runner.run(orchestrator_agent, conversation_context)    print("\n--- Orchestrator Steps ---")    for item in orchestrator_result.new_items:        if isinstance(item, MessageOutputItem):            text = ItemHelpers.text_message_output(item)            if text:                print(f"  - {text}")    synthesizer_result = await Runner.run(synthesizer_agent, orchestrator_result.to_input_list())    # Store response    response_id = str(uuid.uuid4())    with conn.cursor() as cur:        cur.execute("""            INSERT INTO chat_history (id, thread_id, role, message, timestamp)            VALUES (:id, :thread_id, :role, :message, CURRENT_TIMESTAMP)        """, {'id': response_id, 'thread_id': thread_id, 'role': 'assistant', 'message': synthesizer_result.final_output})        conn.commit()    print(f"\n--- Response ---\n{synthesizer_result.final_output}\n")    return synthesizer_result.final_output, thread_id

In [ ]:
def run_research_assistant_chat(query, thread_id=None):    """Synchronous wrapper for the chat function."""    loop = asyncio.new_event_loop()    asyncio.set_event_loop(loop)    result = loop.run_until_complete(research_assistant_chat(query, thread_id))    loop.close()    return result

In [ ]:
def research_chat_session():    """Interactive chat session."""    print("Research Assistant Chat (type 'q' to quit)")    print("=" * 50)    thread_id = None    while True:        user_input = input("\nYou: ").strip()        if user_input.lower() in ('q', 'quit', 'exit'):            print("Session ended.")            break        if not user_input:            continue        result, thread_id = run_research_assistant_chat(user_input, thread_id)

In [ ]:
research_chat_session()

## Session Memory with Oracle AI Database> **Companion guide:** See `docs/part-8-session-memory.md`This section implements a custom `OracleSession` adapter compatible with agent session APIs.

In [ ]:
from typing import List, Optional, Unionfrom datetime import datetimeimport json# ╔══════════════════════════════════════════════════════════════╗# ║  TODO: Implement OracleSession                               ║# ║                                                              ║# ║  Build a class with these async methods:                     ║# ║  - get_items(limit) — retrieve items ordered by timestamp    ║# ║  - add_items(items) — store items as JSON in CLOB column     ║# ║  - pop_item(limit) — remove and return most recent item(s)   ║# ║  - clear_session() — delete all items for this session       ║# ║                                                              ║# ║  Use self.session_id as thread_id for filtering.             ║# ║  Items are stored as JSON strings in the message column.     ║# ║                                                              ║# ║  See: docs/part-8-session-memory.md                         ║# ╚══════════════════════════════════════════════════════════════╝class OracleSession:    """Custom Oracle session implementation following the Session protocol."""    def __init__(self, session_id: str, connection, table_name: str = "chat_history"):        self.session_id = session_id        self.conn = connection        self.table_name = table_name    async def get_items(self, limit: Optional[int] = None) -> List[dict]:        """Retrieve conversation history for this session."""        pass  # YOUR CODE HERE    async def add_items(self, items: List[dict]) -> None:        """Store new items for this session."""        pass  # YOUR CODE HERE    async def pop_item(self, limit: Optional[int] = None) -> Optional[Union[dict, List[dict]]]:        """Remove and return the most recent item(s)."""        pass  # YOUR CODE HERE    async def clear_session(self) -> None:        """Clear all items for this session."""        pass  # YOUR CODE HERE

### Basic Example: Agent with Persistent Session MemoryThese turns demonstrate memory behavior:1. Introduce a user identity and topic2. Ask follow-up questions that depend on prior context3. Remove items to test forgetting4. Clear the session and verify memory reset

In [ ]:
research_agent = Agent(    name="Assistant",    instructions="Research the topic and return the most relevant information. Be concise.",    model=OPENAI_MODEL,)

In [ ]:
session = OracleSession(    session_id="conversation_123",    connection=conn,    table_name="chat_history")

In [ ]:
# First turnresult_from_research_agent = await Runner.run(    starting_agent=research_agent,    input="Hi my name is Nick and I want to learn about papers on optimization",    session=session,)print(result_from_research_agent.final_output)

In [ ]:
# Second turnresult_from_research_agent = await Runner.run(    starting_agent=research_agent,    input="What is a paper about in the results?",    session=session,)print(result_from_research_agent.final_output)

In [ ]:
# Third turn - agent should remember previous contextresult_from_research_agent = await Runner.run(    starting_agent=research_agent,    input="What is my name?",    session=session,)print(result_from_research_agent.final_output)

In [ ]:
# Fourth turn - continuingresult_from_research_agent = await Runner.run(    starting_agent=research_agent,    input="Tell me more about optimization techniques mentioned earlier",    session=session,)print(result_from_research_agent.final_output)

In [ ]:
# Pop the last conversation item to test forgettingawait session.pop_item()

In [ ]:
# Fifth turn: agent should not remember the most recent exchangeresult_from_research_agent = await Runner.run(    starting_agent=research_agent,    input="What was the last thing we discussed?",    session=session,)print(result_from_research_agent.final_output)

In [ ]:
# The agent should still remember our name from the introductionresult_from_research_agent = await Runner.run(    starting_agent=research_agent,    input="What is my name?",    session=session,)print(result_from_research_agent.final_output)

In [ ]:
# Clear the sessionawait session.clear_session()

In [ ]:
# After clearing, the agent should not remember anythingresult_from_research_agent = await Runner.run(    starting_agent=research_agent,    input="What is my name?",    session=session,)print(result_from_research_agent.final_output)